# TickNet-L (From Final) Mid224 — Label Smoothing 0.10

Notebook tự động `git clone` nhánh `release/midterm-model-l-selected` từ GitHub (`https://github.com/khuonglaptri3/TickNets.git`) và chạy thử nghiệm model cuối kỳ thu gọn (`model_l_from_final.py`, 10 PDP blocks `[1, 2, 3, 3, 1]`, `1.28M` params, `0.942 GFLOPs` ở `224×224`) với công thức tốt nhất của `Mid224`:

- `l_from_final_mid224_smoothing010` — `configs/midterm/l_from_final_smoothing010.json` (`model = l_from_final`, `lr = 0.1`, `label_smoothing = 0.10`)

Trước **Run All**:
1. Bật **GPU** và bật **Internet** trong Kaggle Notebook Settings.
2. Không bắt buộc attach dataset thủ công: nếu `/kaggle/input` chưa có `Mid224`, notebook tự tải `data/Mid224` trực tiếp từ GitHub khi clone.
3. Lịch cosine luôn có tổng `EPOCHS = 200`:
   - Phiên 1: giữ `STOP_AFTER = 100`, `RESTORE_ARCHIVE = None` (hoặc đổi `STOP_AFTER = None` nếu muốn chạy thẳng 200 epoch trong 1 phiên).
   - Phiên 2 (nếu chia 2 phiên): Add Input dataset tạo từ file ZIP output của phiên 1, đặt `STOP_AFTER = None` và `RESTORE_ARCHIVE = 'ticknet_l_from_final_mid224_smoothing010_runs_artifacts.zip'`. Khi hoàn tất epoch 200, notebook tự đánh giá `best_val.pt` trên tập `test`.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/khuonglaptri3/TickNets.git'
BRANCH = 'release/midterm-model-l-selected'
REVISION = None  # Tự động lấy commit HEAD mới nhất sau khi git clone.
INPUT_ROOT = Path('/kaggle/input')
REPO = Path('/kaggle/working/TickNets')
DATA = Path('/kaggle/working/mid_experiment_data')
RUNS = Path('/kaggle/working/ticknet_l_from_final_mid224_smoothing010_runs')
RUN_SPECS = (
    {
        'variant': 'Mid224',
        'config': 'configs/midterm/l_from_final_smoothing010.json',
        'run_name': 'l_from_final_mid224_smoothing010',
    },
)
EPOCHS = 200
STOP_AFTER = 100  # Phiên 1 dừng ở epoch 100; phiên 2 (hoặc chạy 1 lèo) đổi thành None.
NUM_WORKERS = 2
SEED = 42
RESTORE_ARCHIVE = None  # None cho phiên đầu; phiên sau nhập tên ZIP hoặc thư mục artifact đã giải nén.
EVALUATE_TEST_WHEN_COMPLETE = True


In [ ]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path, PurePosixPath

CLASSES = ('bird', 'cat', 'dog', 'frog', 'horse')
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.ppm', '.pgm'}


def has_variant_in_root(search_root, variant):
    search_root = Path(search_root)
    if not search_root.exists():
        return False
    for train in search_root.rglob('train'):
        root = train.parent
        if root.name.lower() != variant.lower():
            continue
        if all((root / split / name).is_dir() for split in ('train', 'test') for name in CLASSES):
            return True
    return False


def clone_or_locate_repo(repo_dir, *, repo_url, branch, variants, input_root=Path('/kaggle/input')):
    """Clone branch from GitHub (fetching data/ only if missing from /kaggle/input)."""
    repo_dir = Path(repo_dir)
    need_repo_data = not all(has_variant_in_root(input_root, v) for v in variants)
    if not (repo_dir / '.git').exists():
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        subprocess.run(
            [
                'git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
                '--branch', branch, repo_url, str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth', '1', 'origin', branch], check=True)

    sparse_rules = ['/*', '!/checkpoints/', '!/docs/results/']
    if not need_repo_data:
        sparse_rules.append('!/data/')
    subprocess.run(
        ['git', '-C', str(repo_dir), 'sparse-checkout', 'set', '--no-cone', *sparse_rules],
        check=True,
    )
    subprocess.run(['git', '-C', str(repo_dir), 'checkout', '-B', branch, f'origin/{branch}'], check=True)
    return repo_dir


def plan_batches(run_specs, gpu_count):
    if not run_specs:
        raise ValueError('RUN_SPECS must not be empty')
    slots = max(1, min(int(gpu_count), 2))
    return [
        [(spec, slot) for slot, spec in enumerate(run_specs[start:start + slots])]
        for start in range(0, len(run_specs), slots)
    ]


def build_command(spec, output, *, data_root, epochs, stop_after, seed, num_workers):
    output = Path(output)
    last = output / 'last.pt'
    if output.exists() and not last.is_file():
        raise RuntimeError(f'{output} exists without last.pt; restore a valid artifact or use a clean session')
    command = [
        sys.executable, 'train_mid_experiment.py',
        '--config', spec['config'],
        '--variant', spec['variant'],
        '--data-root', str(data_root),
        '--output-dir', str(output),
        '--epochs', str(epochs),
        '--seed', str(seed),
        '--num-workers', str(num_workers),
        '--device', 'cuda:0',
    ]
    if stop_after is not None:
        command += ['--stop-after-epoch', str(stop_after)]
    if last.is_file():
        command += ['--resume', str(last)]
    return command


def archive_runs(runs):
    runs = Path(runs)
    if not runs.is_dir():
        raise FileNotFoundError(f'Run directory does not exist: {runs}')
    published = runs.parent / f'{runs.name}_artifacts.zip'
    temporary_base = runs.parent / f'.{runs.name}_artifacts.tmp'
    temporary_zip = Path(str(temporary_base) + '.zip')
    try:
        generated = Path(shutil.make_archive(
            str(temporary_base), 'zip', root_dir=runs.parent, base_dir=runs.name,
        ))
        generated.replace(published)
    finally:
        temporary_zip.unlink(missing_ok=True)
    return published


def restore_runs(source, runs):
    source, runs = Path(source), Path(runs)
    if runs.exists():
        raise FileExistsError(f'Refusing to overwrite existing run directory: {runs}')
    if source.is_dir():
        source_runs = source if source.name == runs.name else source / runs.name
        if not source_runs.is_dir():
            raise ValueError(f'Artifact directory does not contain {runs.name!r}: {source}')
        shutil.copytree(source_runs, runs)
        return
    if not source.is_file() or not zipfile.is_zipfile(source):
        raise ValueError(f'Not a ZIP or extracted artifact directory: {source}')
    root = runs.parent.resolve()
    runs.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as handle:
        for info in handle.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or '..' in member.parts or not member.parts or member.parts[0] != runs.name:
                raise ValueError(f'unsafe ZIP member: {info.filename!r}')
            destination = (root / Path(*member.parts)).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f'unsafe ZIP member: {info.filename!r}')
        handle.extractall(root)
    if not runs.is_dir():
        raise ValueError(f'Artifact did not contain expected root directory {runs.name!r}')


def resolve_restore_archive(reference, runs, input_root=Path('/kaggle/input')):
    candidate = Path(reference)
    if candidate.exists():
        return candidate
    matches = sorted(input_root.rglob(str(reference)))
    if not matches and str(reference).lower().endswith('.zip'):
        matches = sorted(path for path in input_root.rglob(Path(runs).name) if path.is_dir())
    if len(matches) != 1:
        raise ValueError(
            f'Expected one ZIP or extracted restore directory matching {reference!r}; found {matches}'
        )
    return matches[0]


def locate_variant(input_root, variant):
    candidates = []
    for train in Path(input_root).rglob('train'):
        root = train.parent
        if root.name.lower() != variant.lower():
            continue
        if all((root / split / name).is_dir() for split in ('train', 'test') for name in CLASSES):
            candidates.append(root)
    if len(candidates) != 1:
        raise ValueError(f'Expected exactly one {variant} dataset; found {candidates}')
    return candidates[0]


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def prepare_data(input_root, data_root, variants):
    from PIL import Image

    data_root = Path(data_root)
    data_root.mkdir(parents=True, exist_ok=True)
    rows, memberships = [], {}
    for variant in sorted(set(variants)):
        source = locate_variant(input_root, variant)
        size = 32 if variant == 'Mid32' else 224
        keys = {}
        for split, expected in (('train', 5000), ('test', 50)):
            keys[split] = []
            for class_name in CLASSES:
                images = sorted(
                    path for path in (source / split / class_name).iterdir()
                    if path.suffix.lower() in EXTENSIONS
                )
                if len(images) != expected:
                    raise ValueError((variant, split, class_name, len(images), expected))
                for image_path in images:
                    with Image.open(image_path) as image:
                        actual_size = image.size
                        image.verify()
                    if actual_size != (size, size):
                        raise ValueError((variant, image_path, actual_size, (size, size)))
                    key = f'{class_name}/{image_path.name}'
                    keys[split].append(key)
                    rows.append({
                        'variant': variant,
                        'split': split,
                        'class_name': class_name,
                        'filename': image_path.name,
                        'size_bytes': image_path.stat().st_size,
                        'sha256': file_sha256(image_path),
                    })
        if set(keys['train']) & set(keys['test']):
            raise ValueError(f'{variant} has class/filename overlap between train and test')
        memberships[variant] = keys
        link = data_root / variant
        if link.exists() or link.is_symlink():
            if link.resolve() != source.resolve():
                raise ValueError(f'{link} points to a different dataset; restart a clean session')
        else:
            link.symlink_to(source, target_is_directory=True)
    if len(memberships) == 2 and memberships['Mid32'] != memberships['Mid224']:
        raise ValueError('Paired Mid32/Mid224 class and filenames do not match')
    manifest = data_root / 'split_manifest.csv'
    with manifest.open('w', encoding='utf-8', newline='') as handle:
        fields = ('variant', 'split', 'class_name', 'filename', 'size_bytes', 'sha256')
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return manifest


def run_batch(batch, *, repo, runs, data_root, epochs, stop_after, seed, num_workers):
    processes = []
    try:
        for spec, physical_gpu in batch:
            output = Path(runs) / spec['run_name']
            command = build_command(
                spec, output, data_root=data_root, epochs=epochs,
                stop_after=stop_after, seed=seed, num_workers=num_workers,
            )
            environment = os.environ.copy()
            environment['CUDA_VISIBLE_DEVICES'] = str(physical_gpu)
            environment['PYTHONUNBUFFERED'] = '1'
            environment['PYTHONDONTWRITEBYTECODE'] = '1'
            print(f"START {spec['run_name']} on physical GPU {physical_gpu}: {' '.join(command)}", flush=True)
            started = time.monotonic()
            process = subprocess.Popen(command, cwd=repo, env=environment)
            processes.append((spec, physical_gpu, process, started))
        failures = []
        for spec, physical_gpu, process, started in processes:
            code = process.wait()
            elapsed = round(time.monotonic() - started, 1)
            print(f"END {spec['run_name']} gpu={physical_gpu} exit={code} seconds={elapsed}", flush=True)
            if code:
                failures.append((spec['run_name'], code))
        if failures:
            raise RuntimeError(f'Training subprocesses failed: {failures}')
    except BaseException:
        for _, _, process, _ in processes:
            if process.poll() is None:
                process.terminate()
        for _, _, process, _ in processes:
            if process.poll() is None:
                process.wait()
        raise


def evaluate_completed_runs(runs, run_specs, *, repo, data_root, target_epochs, num_workers):
    results = []
    for spec in run_specs:
        run = Path(runs) / spec['run_name']
        history_path = run / 'epochs.csv'
        best_ckpt = run / 'best_val.pt'
        if not history_path.is_file() or not best_ckpt.is_file():
            continue
        with history_path.open(encoding='utf-8', newline='') as handle:
            epochs = list(csv.DictReader(handle))
        last_epoch = int(epochs[-1]['epoch']) if epochs else 0
        if last_epoch < target_epochs:
            continue
        eval_dir = run / 'test_eval_best_val'
        if eval_dir.exists():
            shutil.rmtree(eval_dir)
        command = [
            sys.executable, 'train_mid_experiment.py',
            '--config', spec['config'],
            '--variant', spec['variant'],
            '--data-root', str(data_root),
            '--output-dir', str(eval_dir),
            '--evaluate', str(best_ckpt),
            '--num-workers', str(num_workers),
            '--device', 'cuda:0',
        ]
        subprocess.run(command, cwd=repo, check=True)
        eval_json = json.loads((eval_dir / 'test_metrics.json').read_text(encoding='utf-8'))
        results.append(eval_json)
    return results


def summarize_runs(runs, run_specs, target_epochs):
    summary = []
    for spec in run_specs:
        run = Path(runs) / spec['run_name']
        config_path, history_path = run / 'config.json', run / 'epochs.csv'
        eval_path = run / 'test_eval_best_val' / 'test_metrics.json'
        if not config_path.is_file() or not history_path.is_file():
            summary.append({
                'run': spec['run_name'], 'variant': spec['variant'], 'model': '',
                'status': 'not_started', 'last_epoch': 0, 'target_epochs': target_epochs,
                'best_epoch': '', 'val_top1': '', 'val_loss': '',
                'test_top1': '', 'test_loss': '', 'test_macro_f1': '',
                'learning_rate': '', 'label_smoothing': '',
                'parameters': '', 'gflops': '',
                'seed': '', 'revision': '', 'validation_sha256': '',
            })
            continue
        config = json.loads(config_path.read_text(encoding='utf-8'))
        with history_path.open(encoding='utf-8', newline='') as handle:
            epochs = list(csv.DictReader(handle))
        valid = [row for row in epochs if row['val_top1']]
        best = max(valid, key=lambda row: (float(row['val_top1']), -float(row['val_loss']))) if valid else None
        last_epoch = int(epochs[-1]['epoch']) if epochs else 0
        test_eval = json.loads(eval_path.read_text(encoding='utf-8')) if eval_path.is_file() else {}
        summary.append({
            'run': spec['run_name'], 'variant': config['variant'], 'model': config.get('model', ''),
            'status': 'complete' if last_epoch >= target_epochs else 'partial',
            'last_epoch': last_epoch, 'target_epochs': target_epochs,
            'best_epoch': int(best['epoch']) if best else '',
            'val_top1': best['val_top1'] if best else '',
            'val_loss': best['val_loss'] if best else '',
            'test_top1': test_eval.get('top1', ''),
            'test_loss': test_eval.get('loss', ''),
            'test_macro_f1': test_eval.get('macro_f1', ''),
            'learning_rate': config['learning_rate'],
            'label_smoothing': config.get('label_smoothing', 0.0),
            'parameters': config['complexity']['learnable_parameters'],
            'gflops': round(config['complexity']['flops'] / 1e9, 6),
            'seed': config['seed'], 'revision': config['git_revision'],
            'validation_sha256': config['validation_sha256'],
        })
    return summary


In [ ]:
if RESTORE_ARCHIVE is not None:
    restore_path = resolve_restore_archive(RESTORE_ARCHIVE, RUNS, INPUT_ROOT)
    restore_runs(restore_path, RUNS)
    print('Restored:', restore_path, '->', RUNS)

variants = tuple(sorted({spec['variant'] for spec in RUN_SPECS}))
REPO = clone_or_locate_repo(
    REPO, repo_url=REPO_URL, branch=BRANCH, variants=variants, input_root=INPUT_ROOT,
)
actual_revision = subprocess.check_output(
    ['git', '-c', f'safe.directory={REPO.resolve()}', '-C', str(REPO), 'rev-parse', 'HEAD'],
    text=True,
).strip()
if not (REPO / 'models/model_l_from_final.py').is_file():
    raise RuntimeError(f'Nhánh {BRANCH} tại commit {actual_revision} chưa có models/model_l_from_final.py')
for spec in RUN_SPECS:
    if not (REPO / spec['config']).is_file():
        raise RuntimeError(f"Thiếu file config {spec['config']} trong {REPO}")
REVISION = actual_revision

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU trong Kaggle Notebook Settings rồi Run All lại')
print('GitHub Repo:', REPO, 'Branch:', BRANCH)
print('Source commit:', actual_revision)
print('Python:', sys.version)
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
variants = tuple(sorted({spec['variant'] for spec in RUN_SPECS}))
project_data = REPO / 'data'
data_input = INPUT_ROOT if all(has_variant_in_root(INPUT_ROOT, v) for v in variants) else project_data
manifest = prepare_data(data_input, DATA, variants)
manifest_digest = hashlib.sha256(manifest.read_bytes()).hexdigest()

RUNS.mkdir(parents=True, exist_ok=True)
archived_manifest = RUNS / 'source_split_manifest.csv'
if archived_manifest.is_file():
    if archived_manifest.read_bytes() != manifest.read_bytes():
        raise ValueError('Source dataset content manifest changed; refusing resume')
elif any((RUNS / spec['run_name']).exists() for spec in RUN_SPECS):
    raise ValueError('Restored runs are missing source_split_manifest.csv')
else:
    archived_manifest.write_bytes(manifest.read_bytes())

revision_file = RUNS / 'source_revision.txt'
if revision_file.is_file() and revision_file.read_text(encoding='utf-8').strip() != REVISION:
    raise ValueError('Restored artifact was created from a different source revision')
revision_file.write_text(REVISION + '\n', encoding='utf-8')
print('Data input:', data_input)
print('Writable data links:', DATA)
print('Variants:', variants)
print('Content manifest SHA256:', manifest_digest)


In [ ]:
batches = plan_batches(RUN_SPECS, torch.cuda.device_count())
print('Execution batches:', [[(spec['run_name'], gpu) for spec, gpu in batch] for batch in batches])
try:
    for batch in batches:
        run_batch(
            batch, repo=REPO, runs=RUNS, data_root=DATA, epochs=EPOCHS,
            stop_after=STOP_AFTER, seed=SEED, num_workers=NUM_WORKERS,
        )
        print('Checkpoint artifact:', archive_runs(RUNS))
    if EVALUATE_TEST_WHEN_COMPLETE:
        eval_results = evaluate_completed_runs(
            RUNS, RUN_SPECS, repo=REPO, data_root=DATA,
            target_epochs=EPOCHS, num_workers=NUM_WORKERS,
        )
        for item in eval_results:
            print('Test evaluation (best_val.pt):', item)
finally:
    if RUNS.is_dir():
        print('Latest recoverable artifact:', archive_runs(RUNS))


In [ ]:
summary = summarize_runs(RUNS, RUN_SPECS, EPOCHS)
summary_path = RUNS / 'validation_summary.csv'
with summary_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
for row in summary:
    print(row)
print('Download artifact:', archive_runs(RUNS))


## Sau khi train

Tải file `ticknet_l_from_final_mid224_smoothing010_runs_artifacts.zip` trong Kaggle Output. File ZIP bao gồm:
- `best_val.pt`, `last.pt`, `epochs.csv`, `config.json`
- `test_eval_best_val/test_metrics.json`, `confusion_matrix.csv`, `test_predictions.csv` (đánh giá một lần trên tập `test` sau khi hoàn thành đủ 200 epoch)
- `validation_summary.csv` tổng hợp cả `val_top1`, `test_top1`, `test_macro_f1`, `parameters`, và `gflops`.
